# 02 - Camada Silver: tratamento e unificação

Lê a cópia em Parquet da Bronze, padroniza os nomes das colunas, remove duplicidades, valida as contagens, grava uma tabela Silver por edição e uma tabela unificada (`silver_perfil_profissionais`) com as perguntas entre 2023, 2024 e 2025-2026. Todas as tabelas são registradas no database `datahackers_silver`.

## 1. Configuração da sessão

In [5]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2


## 2. Contexto Spark e bibliotecas

In [1]:
import ast
import re
import unicodedata
from functools import reduce

import boto3
from pyspark.context import SparkContext
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Sessão Spark pronta para trabalhar na Silver.")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: ebb67835-d1f5-4c81-9293-d7e4d00dd882
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session ebb67835-d1f5-4c81-9293-d7e4d00dd882 to get into ready status...
Session ebb67835-d1f5-4c81-9293-d7e4d00dd882 has been created.
Sessão Spark pronta para trabalhar na Silver.


## 3. Parâmetros do projeto

In [2]:
BUCKET = "teste-fiap-fase3"
DATABASE_SILVER = "datahackers_silver"

EDICOES = {
    "2023": "state_of_data_2023",
    "2024": "state_of_data_2024",
    "2025-2026": "state_of_data_2025_2026",
}

origens_bronze = {ano: f"s3://{BUCKET}/bronze/parquet/{pasta}/" for ano, pasta in EDICOES.items()}
destinos_silver = {ano: f"s3://{BUCKET}/silver/{pasta}/" for ano, pasta in EDICOES.items()}
DESTINO_UNIFICADA = f"s3://{BUCKET}/silver/perfil_profissionais/"

# Confirmação das quantidades esperadas após a remoção de duplicidades
contagens_esperadas = {
    "2023": {"registros": 5293, "colunas": 400},
    "2024": {"registros": 5215, "colunas": 404},
    "2025-2026": {"registros": 3494, "colunas": 389},
}

## 4. Leitura da Bronze

In [3]:
# Cópia em Parquet da Bronze
bases_bronze = {}
for ano, caminho in origens_bronze.items():
    bases_bronze[ano] = spark.read.parquet(caminho)
    print(f"Bronze {ano:<10} {bases_bronze[ano].count():>6,} registros | {len(bases_bronze[ano].columns)} colunas")

Bronze 2023        5,293 registros | 399 colunas
Bronze 2024        5,217 registros | 403 colunas
Bronze 2025-2026   3,495 registros | 388 colunas


## 5. Funções de padronização dos nomes das colunas

In [4]:
# Padronização de cada nome: minúsculas, sem acentos e com underscores.
def normalizar_nome_coluna(nome_original):
    nome = str(nome_original).strip()
    nome = re.sub(r"#\d+$", "", nome)

    try:
        valor = ast.literal_eval(nome)
        if isinstance(valor, tuple):
            nome = "_".join(str(parte).strip() for parte in valor)
    except (ValueError, SyntaxError):
        pass

    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii")
    nome = nome.lower().strip()
    nome = re.sub(r"[^a-z0-9]+", "_", nome).strip("_")

    if not nome:
        nome = "coluna_sem_nome"
    if nome[0].isdigit():
        nome = f"c_{nome}"

    return nome

# Tirar nomes repetidos
def padronizar_colunas(df):
    nomes_novos = []
    nomes_usados = set()

    for nome_original in df.columns:
        nome_base = normalizar_nome_coluna(nome_original)
        nome_final = nome_base
        contador = 2

        while nome_final in nomes_usados:
            nome_final = f"{nome_base}_{contador}"
            contador += 1

        nomes_usados.add(nome_final)
        nomes_novos.append(nome_final)

    return df.toDF(*nomes_novos)

print("Funções de padronização definidas.")

Funções de padronização definidas.


## 6. Tratamento e validação

In [5]:
# Remover apenas linhas completamente duplicadas e incluindo a coluna ano_pesquisa.
silver_edicoes = {}

for ano, df_bronze in bases_bronze.items():
    df_tratado = (
        padronizar_colunas(df_bronze)
        .dropDuplicates()
        .withColumn("ano_pesquisa", F.lit(ano))
    )
    total_registros = df_tratado.count()
    total_colunas = len(df_tratado.columns)
    removidas = bases_bronze[ano].count() - total_registros
    print(f"{ano:<10} {total_registros:>6,} registros | {total_colunas} colunas | duplicadas removidas: {removidas}")

    esperado = contagens_esperadas[ano]
    if total_registros != esperado["registros"]:
        raise ValueError(f"Base {ano}: esperava {esperado['registros']} registros, encontrei {total_registros}. Não gravei a Silver.")
    if total_colunas != esperado["colunas"]:
        raise ValueError(f"Base {ano}: esperava {esperado['colunas']} colunas, encontrei {total_colunas}. Não gravei a Silver.")

    silver_edicoes[ano] = df_tratado

2023        5,293 registros | 400 colunas | duplicadas removidas: 0
2024        5,215 registros | 404 colunas | duplicadas removidas: 2
2025-2026   3,494 registros | 389 colunas | duplicadas removidas: 1


## 7. Gravação e catalogação das tabelas

In [6]:
# Base tratada em parquet e catalogação
for ano, df_tratado in silver_edicoes.items():
    tabela = f"{DATABASE_SILVER}.silver_{EDICOES[ano]}"
    (
        df_tratado.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .option("path", destinos_silver[ano])
        .saveAsTable(tabela)
    )
    print(f"Silver {ano:<10} gravada em {destinos_silver[ano]} | tabela {tabela}")

Silver 2023       gravada em s3://teste-fiap-fase3/silver/state_of_data_2023/ | tabela datahackers_silver.silver_state_of_data_2023
Silver 2024       gravada em s3://teste-fiap-fase3/silver/state_of_data_2024/ | tabela datahackers_silver.silver_state_of_data_2024
Silver 2025-2026  gravada em s3://teste-fiap-fase3/silver/state_of_data_2025_2026/ | tabela datahackers_silver.silver_state_of_data_2025_2026


## 8. Localização das perguntas

In [7]:
# Mostrando em que coluna cada pergunta aparece
TERMOS = {
    "faixa_idade": "faixa_idade", "genero": "_b_genero", "raca_etnia": "cor_raca",
    "regiao": "regiao_onde_mora", "escolaridade": "nivel_de_ensino", "setor": "_b_setor",
    "cargo": "cargo_atual", "senioridade": "_g_nivel", "faixa_salarial": "faixa_salarial",
    "modelo_trabalho": "trabalho", "uso_ia": "chatgpt", "ia_empresa": "ai_generativa",
}
for campo, termo in TERMOS.items():
    print(f"\n{campo}")
    for ano, df in silver_edicoes.items():
        achadas = [c for c in df.columns if termo in c]
        print(f"  {ano:<10} {achadas[:5]}")


faixa_idade
  2023       ['p1_a_1_faixa_idade']
  2024       ['c_1_a_1_faixa_idade']
  2025-2026  ['c_1_a_1_faixa_idade']

genero
  2023       ['p1_b_genero']
  2024       ['c_1_b_genero']
  2025-2026  ['c_1_b_genero']

raca_etnia
  2023       ['p1_c_cor_raca_etnia', 'p1_e_2_experiencia_prejudicada_devido_a_minha_cor_raca_etnia']
  2024       ['c_1_c_cor_raca_etnia', 'c_1_e_2_sim_devido_a_minha_cor_raca_etnia']
  2025-2026  ['c_1_c_cor_raca_etnia', 'c_1_e_2_sim_devido_a_minha_cor_raca_etnia']

regiao
  2023       ['p1_i_2_regiao_onde_mora']
  2024       ['c_1_i_2_regiao_onde_mora']
  2025-2026  ['c_1_i_2_regiao_onde_mora']

escolaridade
  2023       ['p1_l_nivel_de_ensino']
  2024       ['c_1_l_nivel_de_ensino']
  2025-2026  ['c_1_l_nivel_de_ensino']

setor
  2023       ['p2_b_setor']
  2024       ['c_2_b_setor']
  2025-2026  ['c_2_b_setor']

cargo
  2023       ['p2_f_cargo_atual']
  2024       ['c_2_f_cargo_atual']
  2025-2026  ['c_2_f_cargo_atual']

senioridade
  2023       ['p2_g_n

## 9. Tabela unificada com as perguntas

In [8]:
# Fazendo um DE PARA com as perguntas
DE_PARA = {
    "faixa_idade":      {"2023": "p1_a_1_faixa_idade",   "2024": "c_1_a_1_faixa_idade",   "2025-2026": "c_1_a_1_faixa_idade"},
    "genero":           {"2023": "p1_b_genero",          "2024": "c_1_b_genero",          "2025-2026": "c_1_b_genero"},
    "raca_etnia":       {"2023": "p1_c_cor_raca_etnia",  "2024": "c_1_c_cor_raca_etnia",  "2025-2026": "c_1_c_cor_raca_etnia"},
    "regiao":           {"2023": "p1_i_2_regiao_onde_mora", "2024": "c_1_i_2_regiao_onde_mora", "2025-2026": "c_1_i_2_regiao_onde_mora"},
    "escolaridade":     {"2023": "p1_l_nivel_de_ensino", "2024": "c_1_l_nivel_de_ensino", "2025-2026": "c_1_l_nivel_de_ensino"},
    "setor":            {"2023": "p2_b_setor",           "2024": "c_2_b_setor",           "2025-2026": "c_2_b_setor"},
    "cargo":            {"2023": "p2_f_cargo_atual",     "2024": "c_2_f_cargo_atual",     "2025-2026": "c_2_f_cargo_atual"},
    "senioridade":      {"2023": "p2_g_nivel",           "2024": "c_2_g_nivel",           "2025-2026": "c_2_g_nivel"},
    "faixa_salarial":   {"2023": "p2_h_faixa_salarial",  "2024": "c_2_h_faixa_salarial",  "2025-2026": "c_2_h_faixa_salarial"},
    "modelo_trabalho":  {"2023": "p2_r_atualmente_qual_a_sua_forma_de_trabalho",
                         "2024": "c_2_r_modelo_de_trabalho_atual",
                         "2025-2026": "c_2_q_modelo_de_trabalho_atual"},
    "uso_ia_trabalho":  {"2023": "p4_m_utiliza_chatgpt_ou_llms_no_trabalho",
                         "2024": "c_4_m_usa_chatgpt_ou_copilot_no_trabalho",
                         "2025-2026": "c_4_j_usa_chatgpt_ou_copilot_no_trabalho"},
    "tipo_uso_ia_empresa": {"2023": "p3_f_tipos_de_uso_de_ai_generativa_e_llms_na_empresa",
                            "2024": "c_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa",
                            "2025-2026": "c_3_f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa"},
    "motivos_nao_uso_ia_empresa": {"2023": "p3_g_motivos_que_levam_a_empresa_a_nao_usar_ai_genrativa_e_llms",
                                   "2024": "c_3_g_motivos_para_nao_usar_ai_generativa_e_llm",
                                   "2025-2026": "c_3_h_motivos_para_nao_usar_ai_generativa_e_llm"},
    "resultados_ia_empresa": {"2023": None, "2024": None, "2025-2026": "c_3_g_empresa_est*"},
    # Situação de trabalho (P2_a), atuação como gestor (P2_d) e cargo de gestor (P2_e).
    "situacao_trabalho": {"2023": "p2_a_*", "2024": "c_2_a_*", "2025-2026": "c_2_a_*"},
    "gestor":            {"2023": "p2_d_*", "2024": "c_2_d_*", "2025-2026": "c_2_d_*"},
    "cargo_gestor":      {"2023": "p2_e_*", "2024": "c_2_e_*", "2025-2026": "c_2_e_*"},
}

def resolver_coluna(df, nome, ano, campo):
    if nome is None:
        return None
    if nome.endswith("*"):
        candidatas = [c for c in df.columns if c.startswith(nome[:-1])]
    else:
        candidatas = [c for c in df.columns if c == nome]
    if len(candidatas) != 1:
        raise ValueError(f"{campo} / {ano}: esperava 1 coluna para '{nome}', encontrei {candidatas}.")
    return candidatas[0]

# tirando os espaços em nulo
def limpar_texto(coluna):
    valor = F.trim(F.col(coluna))
    return F.when(valor == "", None).otherwise(valor)

partes = []
for ano, df in silver_edicoes.items():
    colunas = [F.col("ano_pesquisa")]
    for campo, por_ano in DE_PARA.items():
        origem = resolver_coluna(df, por_ano[ano], ano, campo)
        if por_ano[ano] and por_ano[ano].endswith("*"):
            print(f"{campo:<22} {ano:<10} -> {origem}")
        colunas.append((limpar_texto(origem) if origem else F.lit(None).cast("string")).alias(campo))
    partes.append(df.select(*colunas))

df_perfil = reduce(DataFrame.unionByName, partes)
df_perfil.groupBy("ano_pesquisa").count().orderBy("ano_pesquisa").show()

situacao_trabalho      2023       -> p2_a_qual_sua_situacao_atual_de_trabalho
gestor                 2023       -> p2_d_gestor
cargo_gestor           2023       -> p2_e_cargo_como_gestor
situacao_trabalho      2024       -> c_2_a_situacao_de_trabalho
gestor                 2024       -> c_2_d_atua_como_gestor
cargo_gestor           2024       -> c_2_e_cargo_como_gestor
resultados_ia_empresa  2025-2026  -> c_3_g_empresa_esta_conseguindo_ter_bons_resultados_com_llms
situacao_trabalho      2025-2026  -> c_2_a_situacao_de_trabalho
gestor                 2025-2026  -> c_2_d_atua_como_gestor
cargo_gestor           2025-2026  -> c_2_e_cargo_como_gestor
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|        2023| 5293|
|        2024| 5215|
|   2025-2026| 3494|
+------------+-----+


## 10. Faixas salariais: correção de rótulos e limites numéricos

In [9]:
# Correção dos rótulos com erro
CORRECOES_FAIXA = {
    "de R$ 25.001/mês a R$ 3000/mês": "de R$ 25.001/mês a R$ 30.000/mês",
    # Variação de rótulo com 1 resposta em 2023; corresponde à faixa de R$ 1.001 a R$ 2.000.
    "de R$ 101/mês a R$ 2.000/mês": "de R$ 1.001/mês a R$ 2.000/mês",
}
df_perfil = df_perfil.replace(CORRECOES_FAIXA, subset=["faixa_salarial"])

# Limites numéricos de cada faixa
esquema_limites = T.StructType([
    T.StructField("salario_min", T.IntegerType()),
    T.StructField("salario_max", T.IntegerType()),
])

@F.udf(esquema_limites)
def limites_faixa(faixa):
    if faixa is None:
        return (None, None)
    numeros = [int(n.strip(".").replace(".", "")) for n in re.findall(r"\d[\d.]*", faixa)]
    texto = faixa.lower()
    if not numeros:
        return (None, None)
    if texto.startswith("menos"):
        return (0, numeros[0])
    if texto.startswith("acima"):
        return (numeros[0], None)
    if len(numeros) >= 2:
        return (numeros[0], numeros[1])
    return (numeros[0], numeros[0])

df_perfil = (
    df_perfil
    .withColumn("limites", limites_faixa("faixa_salarial"))
    .withColumn("salario_min", F.col("limites.salario_min"))
    .withColumn("salario_max", F.col("limites.salario_max"))
    .drop("limites")
)

# Faixas com os limites extraídos
(df_perfil.groupBy("faixa_salarial", "salario_min", "salario_max")
    .pivot("ano_pesquisa").count()
    .orderBy("salario_min").show(40, truncate=False))


+--------------------------------+-----------+-----------+----+----+---------+
|faixa_salarial                  |salario_min|salario_max|2023|2024|2025-2026|
+--------------------------------+-----------+-----------+----+----+---------+
|NULL                            |NULL       |NULL       |540 |354 |267      |
|Menos de R$ 1.000/mês           |0          |1000       |30  |33  |21       |
|de R$ 1.001/mês a R$ 2.000/mês  |1001       |2000       |216 |155 |117      |
|de R$ 2.001/mês a R$ 3.000/mês  |2001       |3000       |288 |237 |130      |
|de R$ 3.001/mês a R$ 4.000/mês  |3001       |4000       |352 |270 |157      |
|de R$ 4.001/mês a R$ 6.000/mês  |4001       |6000       |745 |592 |344      |
|de R$ 6.001/mês a R$ 8.000/mês  |6001       |8000       |637 |656 |367      |
|de R$ 8.001/mês a R$ 12.000/mês |8001       |12000      |1026|1080|707      |
|de R$ 12.001/mês a R$ 16.000/mês|12001      |16000      |650 |716 |484      |
|de R$ 16.001/mês a R$ 20.000/mês|16001      |20000 

## 11. Valores distintos por edição (base para harmonizar as categorias)

In [10]:
# Se as categorias de resposta mudaram entre as edições
for campo in ["senioridade", "modelo_trabalho", "uso_ia_trabalho", "genero", "regiao"]:
    print(f"\n=== {campo} ===")
    (df_perfil.groupBy(campo).pivot("ano_pesquisa").count()
        .orderBy(campo).show(30, truncate=False))


=== senioridade ===
+-------------------+----+----+---------+
|senioridade        |2023|2024|2025-2026|
+-------------------+----+----+---------+
|NULL               |1436|1399|994      |
|Especialista/Staff+|NULL|NULL|349      |
|Júnior             |1046|868 |518      |
|Pleno              |1392|1376|775      |
|Sênior             |1419|1572|858      |
+-------------------+----+----+---------+


=== modelo_trabalho ===
+--------------------------------------------------------------------------------------------------------------+----+----+---------+
|modelo_trabalho                                                                                               |2023|2024|2025-2026|
+--------------------------------------------------------------------------------------------------------------+----+----+---------+
|NULL                                                                                                          |540 |354 |267      |
|Modelo 100% presencial                    

## 11.1 Harmonização das categorias entre as edições

- `gestor`: a pesquisa não pergunta o nível de quem é gestor, por isso a senioridade fica nula para esse grupo.
- `senioridade_harmonizada`: Júnior, Pleno, Sênior+ (Sênior e Especialista/Staff+, categoria criada em 2025-2026) e Gestor.
- `modelo_trabalho_resumido`: rótulos curtos para os gráficos (as categorias são as mesmas nas três edições).
- Indicadores de IA: a pergunta é de múltipla escolha e as opções mudaram de texto entre as edições.
  Cada opção é identificada por um trecho do texto, sem separar por vírgula (as próprias opções têm vírgulas).

In [11]:
# Confiro o formato das respostas de situação de trabalho e de gestor.
for campo in ["situacao_trabalho", "gestor"]:
    print(f"\n=== {campo} ===")
    (df_perfil.groupBy(campo).pivot("ano_pesquisa").count()
        .orderBy(campo).show(20, truncate=False))

# Gestor: converto a resposta em verdadeiro/falso.
VALORES_SIM = ["1", "1.0", "sim", "true", "verdadeiro"]
gestor_texto = F.lower(F.col("gestor"))
df_perfil = df_perfil.withColumn(
    "gestor",
    F.when(gestor_texto.isin(*VALORES_SIM), F.lit(True))
     .when(F.col("gestor").isNotNull(), F.lit(False))
)

# Senioridade harmonizada para comparar as edições.
df_perfil = df_perfil.withColumn(
    "senioridade_harmonizada",
    F.when(F.col("gestor") == True, "Gestor")
     .when(F.col("senioridade").isin("Sênior", "Especialista/Staff+"), "Sênior+")
     .when(F.col("senioridade").isin("Júnior", "Pleno"), F.col("senioridade"))
)

# Rótulos curtos do modelo de trabalho.
modelo = F.lower(F.col("modelo_trabalho"))
df_perfil = df_perfil.withColumn(
    "modelo_trabalho_resumido",
    F.when(modelo.contains("100% presencial"), "Presencial")
     .when(modelo.contains("100% remoto"), "Remoto")
     .when(modelo.contains("dias fixos"), "Híbrido fixo")
     .when(modelo.contains("flexível"), "Híbrido flexível")
)

# Indicadores de uso de IA no trabalho.
# Em 2023 e 2024 a opção de código era "estilo Copilot (inclui ChatGPT Plus)";
# em 2025-2026 passou a ser "IA para código (Copilot, Cursor, Claude Code)". As duas citam o Github Copilot.
uso = F.lower(F.col("uso_ia_trabalho"))
def indicador(condicao):
    return F.when(F.col("uso_ia_trabalho").isNull(), None).otherwise(condicao)

df_perfil = (
    df_perfil
    .withColumn("ia_gratuita",     indicador(uso.contains("apenas soluções gratuitas")))
    .withColumn("ia_paga_empresa", indicador(uso.contains("empresa em que trabalho paga")))
    .withColumn("ia_paga_propria", indicador(uso.contains("próprio bolso")))
    .withColumn("ia_codigo",       indicador(uso.contains("github copilot")))
)
df_perfil = (
    df_perfil
    .withColumn("ia_usa_alguma", indicador(
        F.col("ia_gratuita") | F.col("ia_paga_empresa") | F.col("ia_paga_propria") | F.col("ia_codigo")))
    # Quem marcou "Não utilizo" junto com alguma opção de uso conta como usuário.
    .withColumn("ia_nao_usa", indicador(uso.contains("não utilizo nenhum tipo") & ~F.col("ia_usa_alguma")))
)

# Conferências.
for campo in ["gestor", "senioridade_harmonizada", "modelo_trabalho_resumido"]:
    print(f"\n=== {campo} ===")
    (df_perfil.groupBy(campo).pivot("ano_pesquisa").count()
        .orderBy(campo).show(20, truncate=False))

print("\n=== Indicadores de IA (% sobre quem respondeu a pergunta) ===")
indicadores_ia = ["ia_usa_alguma", "ia_nao_usa", "ia_gratuita", "ia_paga_empresa", "ia_paga_propria", "ia_codigo"]
(df_perfil.filter(F.col("uso_ia_trabalho").isNotNull())
    .groupBy("ano_pesquisa")
    .agg(F.count("*").alias("respostas"),
         *[F.round(100 * F.avg(F.col(c).cast("int")), 1).alias(c) for c in indicadores_ia])
    .orderBy("ano_pesquisa").show(truncate=False))

# Toda resposta de IA precisa cair em "usa" ou "não usa".
sem_classificacao = df_perfil.filter(
    F.col("uso_ia_trabalho").isNotNull() & ~F.col("ia_usa_alguma") & ~F.col("ia_nao_usa"))
print("Respostas de IA sem classificação:", sem_classificacao.count())


=== situacao_trabalho ===
+---------------------------------------------------------------+----+----+---------+
|situacao_trabalho                                              |2023|2024|2025-2026|
+---------------------------------------------------------------+----+----+---------+
|Desempregado e não estou buscando recolocação                  |10  |11  |10       |
|Desempregado, buscando recolocação                             |361 |205 |167      |
|Empreendedor ou Empregado (CNPJ)                               |555 |494 |428      |
|Empregado (CLT)                                                |3604|3782|2278     |
|Estagiário                                                     |207 |186 |126      |
|Freelancer                                                     |51  |41  |37       |
|Prefiro não informar                                           |10  |16  |18       |
|Servidor Público                                               |146 |124 |123      |
|Somente Estudante (graduaç

## 12. Gravação da tabela unificada

In [12]:
(
    df_perfil.write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .partitionBy("ano_pesquisa")
    .option("path", DESTINO_UNIFICADA)
    .saveAsTable(f"{DATABASE_SILVER}.silver_perfil_profissionais")
)
print(f"Tabela unificada gravada em {DESTINO_UNIFICADA}")

Tabela unificada gravada em s3://teste-fiap-fase3/silver/perfil_profissionais/


## 13. Validação final pelo catálogo

In [13]:
# Tabelas pelo catálogo e numero de registros, colunas e duplicidades.
for ano, pasta in EDICOES.items():
    df_validacao = spark.table(f"{DATABASE_SILVER}.silver_{pasta}")
    registros = df_validacao.count()
    duplicidades = registros - df_validacao.dropDuplicates().count()
    status = "OK" if registros == contagens_esperadas[ano]["registros"] and duplicidades == 0 else "DIVERGENTE"
    print(f"silver_{pasta:<26} {registros:>6,} reg | {len(df_validacao.columns)} col | duplicadas: {duplicidades} | {status}")

df_unificada = spark.table(f"{DATABASE_SILVER}.silver_perfil_profissionais")
print(f"\nsilver_perfil_profissionais: {df_unificada.count():,} registros | {len(df_unificada.columns)} colunas")
df_unificada.groupBy("ano_pesquisa").count().orderBy("ano_pesquisa").show()

glue = boto3.client("glue")
for t in glue.get_tables(DatabaseName=DATABASE_SILVER)["TableList"]:
    print(f"{t['Name']:<34} {t['StorageDescriptor']['Location']}")

silver_state_of_data_2023          5,293 reg | 400 col | duplicadas: 0 | OK
silver_state_of_data_2024          5,215 reg | 404 col | duplicadas: 0 | OK
silver_state_of_data_2025_2026     3,494 reg | 389 col | duplicadas: 0 | OK

silver_perfil_profissionais: 14,002 registros | 28 colunas
+------------+-----+
|ano_pesquisa|count|
+------------+-----+
|        2023| 5293|
|        2024| 5215|
|   2025-2026| 3494|
+------------+-----+

silver_perfil_profissionais        s3://teste-fiap-fase3/silver/perfil_profissionais
silver_state_of_data_2023          s3://teste-fiap-fase3/silver/state_of_data_2023
silver_state_of_data_2024          s3://teste-fiap-fase3/silver/state_of_data_2024
silver_state_of_data_2025_2026     s3://teste-fiap-fase3/silver/state_of_data_2025_2026
